# STEP 3 — Exploratory Data Analysis (EDA) & Program Intelligence Analytics

Tahap ini bertujuan untuk mengeksplorasi performa program pemerintah melalui visualisasi interaktif dan analisis KPI strategis.

Analisis yang dilakukan meliputi:
1. Impact Category Distribution
2. Program Effectiveness Distribution
3. Department Performance Ranking
4. Budget vs Effectiveness Analysis
5. ROI Distribution
6. Top 10 High-Impact Programs

Seluruh visualisasi menggunakan dark executive theme untuk menghasilkan tampilan yang profesional dan siap digunakan pada dashboard maupun README.

In [1]:
# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# =========================================================
# 2. LOAD CLEANED DATASET
# =========================================================

df = pd.read_csv(
    "../data/processed/program_impact_cleaned.csv",
    parse_dates=["start_date"]
)

print("Dataset Shape:", df.shape)

# =========================================================
# 3. GLOBAL DARK EXECUTIVE THEME
# =========================================================

EXECUTIVE_TEMPLATE = "plotly_dark"

CHART_LAYOUT = dict(
    template=EXECUTIVE_TEMPLATE,
    paper_bgcolor="#0F172A",
    plot_bgcolor="#0F172A",
    font=dict(
        family="Segoe UI",
        size=13,
        color="#E2E8F0"
    ),
    title=dict(
        font=dict(size=24, color="#F8FAFC"),
        x=0.02
    ),
    margin=dict(l=40, r=40, t=80, b=40)
)

# =========================================================
# 4. CREATE ASSETS DIRECTORY
# =========================================================

Path("../assets").mkdir(parents=True, exist_ok=True)

Dataset Shape: (10000, 26)


In [2]:
# =========================================================
# 3. EXECUTIVE KPI SUMMARY
# =========================================================

total_programs = len(df)
total_budget = df["budget_allocated"].sum()
avg_effectiveness = df["effectiveness_score"].mean()
avg_roi = df["roi_score"].mean()
avg_satisfaction = df["satisfaction_score"].mean()
high_impact_rate = (
    (df["impact_category"] == "High Impact")
    .mean() * 100
)

print("=" * 70)
print("EXECUTIVE PROGRAM IMPACT SUMMARY")
print("=" * 70)
print(f"Total Programs                : {total_programs:,}")
print(f"Total Budget Allocated        : Rp {total_budget:,.0f}")
print(f"Average Effectiveness Score   : {avg_effectiveness:.2f}")
print(f"Average ROI Score             : {avg_roi:.2f}")
print(f"Average Satisfaction Score    : {avg_satisfaction:.2f} / 5.00")
print(f"High Impact Program Rate      : {high_impact_rate:.2f}%")
print("=" * 70)

EXECUTIVE PROGRAM IMPACT SUMMARY
Total Programs                : 10,000
Total Budget Allocated        : Rp 51,788,261,574,851
Average Effectiveness Score   : 80.92
Average ROI Score             : 80.92
Average Satisfaction Score    : 4.09 / 5.00
High Impact Program Rate      : 22.80%


In [3]:
# =========================================================
# 4. IMPACT CATEGORY DISTRIBUTION
# =========================================================

impact_df = (
    df["impact_category"]
    .value_counts()
    .reset_index()
)
impact_df.columns = ["Impact Category", "Count"]

fig = px.pie(
    impact_df,
    names="Impact Category",
    values="Count",
    hole=0.65,
    title="Program Impact Category Distribution",
    color="Impact Category",
    color_discrete_map={
        "High Impact": "#10B981",
        "Moderate Impact": "#F59E0B",
        "Low Impact": "#EF4444"
    }
)

fig.update_layout(**CHART_LAYOUT)
fig.show()

# Save screenshot as:
# assets/impact-category-distribution.png

In [4]:
# =========================================================
# 5. EFFECTIVENESS SCORE DISTRIBUTION
# =========================================================

fig = px.histogram(
    df,
    x="effectiveness_score",
    nbins=40,
    title="Program Effectiveness Score Distribution",
    color_discrete_sequence=["#60A5FA"]
)

fig.update_layout(
    **CHART_LAYOUT,
    xaxis_title="Effectiveness Score",
    yaxis_title="Number of Programs"
)

fig.show()

# Save screenshot as:
# assets/effectiveness-score-distribution.png

In [6]:
# =========================================================
# 6. DEPARTMENT PERFORMANCE RANKING
# =========================================================

department_perf = (
    df.groupby("department")
    .agg({
        "effectiveness_score": "mean",
        "roi_score": "mean",
        "satisfaction_score": "mean"
    })
    .round(2)
    .sort_values("effectiveness_score", ascending=True)
    .reset_index()
)

fig = px.bar(
    department_perf,
    x="effectiveness_score",
    y="department",
    orientation="h",
    title="Average Program Effectiveness by Department",
    color="effectiveness_score",
    color_continuous_scale="Viridis"
)

fig.update_layout(
    **CHART_LAYOUT,
    xaxis_title="Average Effectiveness Score",
    yaxis_title=""
)

fig.show()

# Save screenshot as:
# assets/department-performance-ranking.png

In [7]:
# =========================================================
# 7. BUDGET VS EFFECTIVENESS ANALYSIS
# =========================================================

fig = px.scatter(
    df.sample(min(2000, len(df)), random_state=42),
    x="budget_allocated",
    y="effectiveness_score",
    color="impact_category",
    size="beneficiaries",
    hover_data=[
        "program_name",
        "department",
        "district",
        "roi_score"
    ],
    title="Budget Allocation vs Program Effectiveness",
    color_discrete_map={
        "High Impact": "#10B981",
        "Moderate Impact": "#F59E0B",
        "Low Impact": "#EF4444"
    }
)

fig.update_layout(
    **CHART_LAYOUT,
    xaxis_title="Budget Allocated (IDR)",
    yaxis_title="Effectiveness Score"
)

fig.show()

# Save screenshot as:
# assets/budget-vs-effectiveness.png

In [8]:
# =========================================================
# 8. ROI SCORE DISTRIBUTION
# =========================================================

fig = px.box(
    df,
    x="impact_category",
    y="roi_score",
    color="impact_category",
    title="ROI Score Distribution by Impact Category",
    color_discrete_map={
        "High Impact": "#10B981",
        "Moderate Impact": "#F59E0B",
        "Low Impact": "#EF4444"
    }
)

fig.update_layout(
    **CHART_LAYOUT,
    xaxis_title="Impact Category",
    yaxis_title="ROI Score"
)

fig.show()

# Save screenshot as:
# assets/roi-score-distribution.png

In [9]:
# =========================================================
# 9. TOP 10 HIGH-IMPACT PROGRAMS
# =========================================================

top_programs = (
    df.sort_values("effectiveness_score", ascending=False)
    [
        [
            "program_name",
            "department",
            "district",
            "budget_allocated",
            "beneficiaries",
            "effectiveness_score",
            "roi_score",
            "impact_category"
        ]
    ]
    .head(10)
)

print("=" * 80)
print("TOP 10 HIGH-IMPACT PROGRAMS")
print("=" * 80)

top_programs

TOP 10 HIGH-IMPACT PROGRAMS


,program_name,department,district,budget_allocated,beneficiaries,effectiveness_score,roi_score,impact_category
9172,Healthcare Outreach,Dinas Kesehatan,Bumiaji,7582050204,18569,98.17,98.17,High Impact
3884,Healthcare Outreach,Dinas Kesehatan,Junrejo,9235058532,28590,96.84,96.84,High Impact
1911,Education Assistance,Dinas Pendidikan,Batu,4238550295,38567,96.41,96.41,High Impact
2156,Education Assistance,Dinas Pendidikan,Bumiaji,1806915575,46265,96.17,96.17,High Impact
1869,Public WiFi Expansion,DISKOMINFO,Junrejo,9318680148,17033,95.92,95.92,High Impact
8756,Agricultural Innovation,Dinas Pertanian,Bumiaji,6418306353,29305,95.82,95.82,High Impact
7696,Healthcare Outreach,Dinas Kesehatan,Batu,3855391265,45813,95.76,95.76,High Impact
3950,Healthcare Outreach,Dinas Kesehatan,Batu,9147526091,23673,95.70,95.70,High Impact
6325,Public WiFi Expansion,DISKOMINFO,Bumiaji,1127110756,13807,95.67,95.67,High Impact
2559,Youth Entrepreneurship,Dinas Koperasi,Bumiaji,1605718202,49171,95.46,95.46,High Impact


In [10]:
# =========================================================
# 10. SAVE EXECUTIVE SUMMARY
# =========================================================

summary = {
    "total_programs": int(total_programs),
    "total_budget_allocated": float(total_budget),
    "average_effectiveness_score": round(avg_effectiveness, 2),
    "average_roi_score": round(avg_roi, 2),
    "average_satisfaction_score": round(avg_satisfaction, 2),
    "high_impact_program_rate": round(high_impact_rate, 2),
    "top_department": (
        department_perf
        .sort_values("effectiveness_score", ascending=False)
        .iloc[0]["department"]
    ),
    "top_program": top_programs.iloc[0]["program_name"]
}

import json
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

with open(
    "../data/processed/program_impact_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4)

print("Executive summary saved successfully.")

Executive summary saved successfully.
